# Intro to Using the NetHack Learning Dataset

There are two different sets of trajectories included in the NetHack Learning Dataset:
- **NLD-NAO**: state-only trajectories from 1.5 M human games played on nethack.alt.org
- **NLD-AA**: state-action-score trajectories from 100k NLE games played by the symbolic-bot winner of the 2021 NetHack Challenge

We also supply a small "taster" dataset, for quick iteration and playing around:
- **NLD-AA-Taster**: ~2,000 randomly chosen trajectories from **NLD-AA**

These trajectories can be used with the `TtyrecDataset` tool which allows for efficiently training on the datasets.  This tutorial describes how to create and use and visualize the dataset, using **NLD-AA-Taster**.



## Downloading the Data

For the time being data is available through various WeTransfer links in the DATASET.md file. Although generally this requires a browser to interface to download, it is also possible to use the command line (see here).

In this case, we use a publically available unzipped version of **NLD-AA-Taster** available on GitHub (or you can access the [zipped release]()).


## Setting up the Database

Adding datasets is easy - all you need is the path to the unzipped directory.

**NOTE** We call different functions to add trajectories generated by NLE (such as **NLD-AA**, **NLD-AA-Taster** or your own dataset) versus those generated from NAO (**NLD-NAO**).  

In [2]:
import nle.dataset as nld

In [3]:
# 1. Get the paths for your unzipped datasets
path_to_nld_aa_taster = "./nld-aa-taster/nle_data"

# 2. Chose a database name/path. By default, most methods with use nld.db.DB (='ttyrecs.db')
dbfilename = "ttyrecs.db"

if not nld.db.exists(dbfilename):
    # 3. Create the db and add the directory
    nld.db.create(dbfilename)
    nld.add_nledata_directory(path_to_nld_aa_taster, "taster-dataset", dbfilename)


# NB: To add the NLE-AA data, or any data generated from nle, use `add_nledata_directory`.
# nld.add_nledata_directory(path_to_nld_aa, "nld-aa", dbfilename)

# NB: To add the NLE-NAO data, use the `add_altorg_directory`.
# nld.add_altorg_directory(path_to_nld_nao, "nld-nao", dbfilename)


You can inspect the dataset using the database tooling:

In [4]:
# Create a connection to specify the database to use
db_conn = nld.db.connect(filename=dbfilename)

# Then you can inspect the number of games in each dataset:
print(f"NLD AA \"Taster\" Dataset has {nld.db.count_games('taster-dataset', conn=db_conn)} games.")

NLD AA "Taster" Dataset has 1934 games.


## Visualizing the Data

Next, to actually load the games for training you'll use the `TtyrecDataset` object:

In [5]:
dataset = nld.TtyrecDataset(
    "taster-dataset",
    batch_size=32,
    seq_length=32,
    dbfilename=dbfilename,
)

This dataset above will return batches of 128 trajectories, returning sequential chunks of length 32.   That is, assuming the length of all trajectories is >>64, the first batch will give timesteps 0-31 of 128 games and the second batch will provide timesteps 32-63 for the same games, etc.

### Whats in the Observation?

In [8]:
minibatch = next(iter(dataset))
minibatch.keys()

dict_keys(['tty_chars', 'tty_colors', 'tty_cursor', 'timestamps', 'done', 'gameids', 'keypresses', 'scores'])

The observation is made up of three components:
- `tty_chars` is a (batched) 2D np.array of the characters displayed at each point on the screen with shape: `[Batch, Time, H, W]`
- `tty_colors` is the associated colors for those characters
- `tty_cursor` provides the cursor position (NOTE: it's not always on the hero!)

These can be easily visualized usign the `tty_render` utility:

In [9]:
from nle.nethack import tty_render

In [10]:
batch_idx = 0
time_idx = 0
chars = minibatch['tty_chars'][batch_idx, time_idx]
colors = minibatch['tty_colors'][batch_idx, time_idx]
cursor = minibatch['tty_cursor'][batch_idx, time_idx]

print(tty_render(chars, colors, cursor))


Salutations Agent, welcome to NetHack!  You are a lawful female human Knight.   
                                                                                
                                                                                
                                                                                
   ------                                                                       
   |..u.|                                                                       
   |.@..|                                                                       
   |....|                                                                       
   |....|                                                                       
   |....|                                                                       
   --+-+-                                                                       
                                                                                
                           

Then, the other elements of the batch are:
- `gameids`: The gameid for the game which the observation is from.
- `timestamps`: The time when the state was recorded, allowing you to understand how long the player took between frames.
- `keypresses`: The keypresses entered after seeing the observation at this timestep (which produces the observation at the next timestep).
- `scores`: The in-game score at this timestep (the result of the action at the previous timestep)
- `done`: Whether the gameid corresponding to the previous timestep's observation completed. If done is `True` this means that the observation at the current timestep is the beginning of the next gameid.

### Converting Actions from Keypresses to Environment Action Space

Note that the "actions" data is actually a keypress (eg ascii) entered not an action value corresponding to the actions in the nle environment.  To convert from keypresses to the action_space of the environment you can use an embedding as shown below:

In [11]:
import torch
from nle.env.tasks import NetHackChallenge


env = NetHackChallenge(
    savedir=None,  # Do not save any recordings.
    character='@', # Randomly rotate through characters.
)

# Then use the environment actions to convert the keypresses.
embed_actions = torch.zeros((256, 1))
for i, a in enumerate(env.actions):
    embed_actions[a.value][0] = i

embed_actions = torch.nn.Embedding.from_pretrained(embed_actions)
keypresses = torch.Tensor(minibatch["keypresses"]).long()
actions = embed_actions(keypresses).squeeze(-1).long()

## Dataset Configuration Options
`shuffle`: While states within a trajectory are always returned sequentially, it is possible to turn on shuffling of the *gameids*.  When true, the order of the gameids sampled is shuffled but not the order of the `seq_length` chunks returned within a single gameid.

`loop_forever`: It is possible to have the iterator loop forever instead of cycling only through the dataset once.

`gameids`: You can specify a list of gameids to return instead of iterating through the full dataset.

`subselect_sql`: And, you can select even more complicated sets of games using specific sql queries.

**NB** A `gameid` of 0 indicates that that index is padded (with 0's).

**Example 1:** Lets create a small dataset of just 4 games, and see the shuffle functionality:

In [9]:
shuffle_small_dataset = nld.TtyrecDataset(
    "taster-dataset",
    batch_size=2,
    seq_length=6000,
    dbfilename=dbfilename,
    shuffle=True,
    loop_forever=False,
    gameids=[34,550,45],
)
for epoch in range(3):
    print(f"Epoch: {epoch}")
    for ind, mb in enumerate(shuffle_small_dataset):
        gameids = mb["gameids"][:, 0]
        print(f"  Batch {ind} first timestep gameids: {gameids}")
    print()


Epoch: 0
  Batch 0 first timestep gameids: [ 45 550]
  Batch 1 first timestep gameids: [ 45 550]
  Batch 2 first timestep gameids: [ 45 550]
  Batch 3 first timestep gameids: [ 45 550]
  Batch 4 first timestep gameids: [ 45 550]
  Batch 5 first timestep gameids: [ 45 550]
  Batch 6 first timestep gameids: [ 45 550]
  Batch 7 first timestep gameids: [ 45 550]
  Batch 8 first timestep gameids: [ 34 550]
  Batch 9 first timestep gameids: [ 34 550]
  Batch 10 first timestep gameids: [  0 550]

Epoch: 1
  Batch 0 first timestep gameids: [34 45]
  Batch 1 first timestep gameids: [34 45]
  Batch 2 first timestep gameids: [34 45]
  Batch 3 first timestep gameids: [550  45]
  Batch 4 first timestep gameids: [550  45]
  Batch 5 first timestep gameids: [550  45]
  Batch 6 first timestep gameids: [550  45]
  Batch 7 first timestep gameids: [550  45]
  Batch 8 first timestep gameids: [550   0]
  Batch 9 first timestep gameids: [550   0]
  Batch 10 first timestep gameids: [550   0]
  Batch 11 first 

**Example 2:** We can train just on the data from a specific character, such as "mon-hum-neu-mal" by using the subselect_sql:

In [19]:
# Build the subselect sql query
subselect_sql = "SELECT gameid FROM games WHERE role=? AND race=?"
subselect_sql_args = ("Mon", "Hum")
batch_size = 10

# Build the dataset
monk_dataset = nld.TtyrecDataset(
    "taster-dataset",
    batch_size=batch_size,
    seq_length=2,
    dbfilename=dbfilename,
    subselect_sql=subselect_sql,
    subselect_sql_args=subselect_sql_args
)

# See from the error how there are fewer than 10k games despite the full dataset having 109k
print(f"Full Dataset has {nld.db.count_games('taster-dataset', conn=db_conn):,} games.")
print(f"Human Monk Subdataset Has: {len(monk_dataset._gameids)} games")

mb = next(iter(monk_dataset))

batch_idx = 0
time_idx = 0
chars = mb['tty_chars'][batch_idx, time_idx]
colors = mb['tty_colors'][batch_idx, time_idx]
cursor = mb['tty_cursor'][batch_idx, time_idx]

print(tty_render(chars, colors, cursor))

Full Dataset has 1,934 games.
Human Monk Subdataset Has: 142 games

Hello Agent, welcome to NetHack!  You are a lawful female human Monk.           
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                         

**Example 3**: Using a threadpool
You can also use a threadpool with the dataset which will speed it up considerably!

In [21]:
from concurrent.futures import ThreadPoolExecutor
import time


with ThreadPoolExecutor(max_workers=10) as tp:
    dataset = nld.TtyrecDataset(
        "taster-dataset",
        batch_size=100,
        seq_length=100,
        dbfilename=dbfilename,
        threadpool=tp
    )
    start = time.time()
    for i, mb in enumerate(dataset):
        if i == 10:
            break
    end = time.time()
    chars = mb['tty_chars'][batch_idx, time_idx]
    colors = mb['tty_colors'][batch_idx, time_idx]
    cursor = mb['tty_cursor'][batch_idx, time_idx]

    print(tty_render(chars, colors, cursor))
# NB this might be v slow on free Colab, try on laptop or server.
print(f"Loaded 100,000 frames in {end-start:.2f}s")


Showing known terrain and traps only...  (For instructions type a '?')          
                                                                                
                          -----------    ------                                 
              --------    |....{....|  ##.....|           -----------           
             #........####...........####|.....###        |.........|           
             #|......| #  |.........|  ##.....|  #########-.........|           
 --------    #-------- #  -----.-----  # |.....#########  |.........|           
 |......-############################### ------        ###|.........|           
 |......|    #  ##           # #                         #..........|           
 |.......########            ###                          -----------           
 ------.-     #              ##                                                 
       #      #              #                                                  
       #      ##           

**Example 4:** Getting Metadata

In [23]:
dataset = nld.TtyrecDataset('taster-dataset', dbfilename=dbfilename)
mb = next(iter(dataset))
gameid = mb["gameids"][0][0]

chars = mb['tty_chars'][0, 1]
colors = mb['tty_colors'][0, 1]
cursor = mb['tty_cursor'][0, 1]

print(tty_render(chars, colors, cursor))

dict(dataset.get_meta(gameid))


                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
                                                   --------                     
                                                   |......|                     
                                                   |.$....|                     
                                                   +...f..|                     
                                                   |..@...+                     
                                                   --------                     
                                                                                
                           

{'gameid': 1357,
 'version': '3.6.6',
 'points': 7021,
 'deathdnum': 0,
 'deathlev': 3,
 'maxlvl': 4,
 'hp': 0,
 'maxhp': 54,
 'deaths': 1,
 'deathdate': 20220518,
 'birthdate': 20220518,
 'uid': 1185200751,
 'role': 'Pri',
 'race': 'Elf',
 'gender': 'Fem',
 'align': 'Cha',
 'name': 'Agent',
 'death': 'died of starvation while praying',
 'conduct': '0xf80',
 'turns': 27155,
 'achieve': '0x0',
 'realtime': 150,
 'starttime': 1652877727,
 'endtime': 1652877877,
 'gender0': 'Fem',
 'align0': 'Cha',
 'flags': '0x4'}

**Example 5** Generating and loading a custom dataset.

In [ ]:
import gym
import nle
import nle.dataset as nld
from datetime import datetime

def generate_rollouts(env):
    obs = env.reset()
    episodes = 0
    while episodes < 10:
        obs, reward, done, info = env.step(env.action_space.sample())
        if done:
            env.reset()
            episodes += 1

# 1. Create some envs, with a savedir directory 'path/to/save/X'
envA = gym.make("NetHackChallenge-v0", savedir="path/to/save/A", save_ttyrec_every=2)
envB = gym.make("NetHackScore-v0", character="Mon-Hum-Neu-Mal", savedir="path/to/save/B", save_ttyrec_every=1)

# 2. Generate rollouts
generate_rollouts(envA)
generate_rollouts(envB)

# 3. Add to directory, with given unique dataset name
name = f"dataset_{datetime.now().time()}"
if not nld.db.exists():
    nld.db.create()
nld.add_nledata_directory("path/to/save", name)

# 4. Use and enjoy!
dataset = nld.TtyrecDataset(name)
print(f"Dataset has {len(dataset._gameids)} entries!")



Adding dataset 'dataset_15:38:53.943302' ('path/to/save') to 'ttyrecs.db' 
Updated 'ttyrecs.db' in 0.00 sec. Size: 0.65 MB, Games: 15
Dataset has 15 entries!


**Example 6:** Use doctstrings - don't forget a lot of the classes and methods have docstrings. Have fun!

In [ ]:
help(nld.TtyrecDataset)

Help on class TtyrecDataset in module nle.dataset.dataset:

class TtyrecDataset(builtins.object)
 |  TtyrecDataset(dataset_name, batch_size=128, seq_length=32, rows=24, cols=80, dbfilename='ttyrecs.db', threadpool=None, gameids=None, shuffle=True, loop_forever=False, subselect_sql=None, subselect_sql_args=None)
 |  
 |  Dataset object to allow iteration through the ttyrecs found in our ttyrec
 |  database.
 |  
 |  Methods defined here:
 |  
 |  __init__(self, dataset_name, batch_size=128, seq_length=32, rows=24, cols=80, dbfilename='ttyrecs.db', threadpool=None, gameids=None, shuffle=True, loop_forever=False, subselect_sql=None, subselect_sql_args=None)
 |      An iterable dataset to load minibatches of NetHack games from compressed
 |      ttyrec*.bz2 files into numpy arrays. (shape: [batch_size, seq_length, ...])
 |      
 |      This class makes use of a sqlite3 database at `dbfilename` to find the
 |      metadata and the location of files in a dataset. It then uses these to
 |   